In [1]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(r"C:\Users\91964\OneDrive\Desktop\Rag_Doc\Report_Format_CSP_Final.pdf")
data = loader.load() 

In [2]:
len(data)

30

In [3]:
from langchain.text_splitter import RecursiveCharacterTextSplitter


text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000)
docs = text_splitter.split_documents(data)


print("Total number of documents: ",len(docs))

Total number of documents:  49


In [4]:
docs[7]

Document(metadata={'producer': 'Microsoft® Word 2021', 'creator': 'Microsoft® Word 2021', 'creationdate': '2024-11-19T10:39:00+05:30', 'author': 'Sheik Ariffa Begum', 'moddate': '2024-11-19T10:39:00+05:30', 'source': 'C:\\Users\\91964\\OneDrive\\Desktop\\Rag_Doc\\Report_Format_CSP_Final.pdf', 'total_pages': 30, 'page': 4, 'page_label': '5'}, page_content='harvesting, and supporting components for efficient operation and \nreal-time navigation. \nEngineering standards and realistic constraints in these areas \nArea Codes & Standards / Realistic Constraints Tick ✓ \nManufacturability \nThis constraint mainly occurs with hardware \nManufacturing, Which integrates the entire system into one \nmicrocontroller, the hardware sensor must be adaptive for \ndifferent sizes of shoes based on the requirements.    \n \nTechnical Constraints \nThere are many exist solutions, but the connection with \nparental control and Arduino Uno has limited \ncomputational capacity, making it challenging to run 

In [1]:
from langchain_chroma import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings

from dotenv import load_dotenv
load_dotenv() 


embeddings = GoogleGenerativeAIEmbeddings(model="models/embedding-001")
vector = embeddings.embed_query("hello, world!")
vector[:5]


[0.05168594419956207,
 -0.030764883384108543,
 -0.03062233328819275,
 -0.02802734263241291,
 0.01813093200325966]

In [4]:
vectorstore = Chroma(
    persist_directory="chroma_db",
    embedding_function=embeddings
)


In [5]:
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 10})

retrieved_docs = retriever.invoke("what is project, Explain.?")


In [6]:
len(retrieved_docs)

0

In [10]:
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 10})

In [11]:
query = "What is project, Explain.?"
retrieved_docs = retriever.invoke(query)

In [12]:
if retrieved_docs:
    print("✅ Number of retrieved documents:", len(retrieved_docs))
    print("\n📄 Top Document Content:\n")
    print(retrieved_docs[0].page_content)
else:
    print("❌ No relevant documents found for the query.")

❌ No relevant documents found for the query.


In [13]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-1.5-pro",temperature=0.3, max_tokens=500)


In [14]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}"),
    ]
)


In [15]:
question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)


In [16]:
response = rag_chain.invoke({"input": "what is project, Explain.?"})
print(response["answer"])


A project is a temporary endeavor undertaken to create a unique product, service, or result.  It has a defined beginning and end, and is often constrained by time, budget, and resources. Projects are distinct from ongoing operations as they are temporary and produce a deliverable.
